In [1]:
import numpy as np
import pyvista as pv
import ufl
from dolfinx import default_scalar_type, fem, mesh, plot
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.nls.petsc import NewtonSolver
from mpi4py import MPI


In [2]:
# Params

Lx = 4
Ly = 2
Lz = 2
nx = 40
ny = 20
nz = 20
tdim = 3

C10 = 0.8
C01 = 0.5
D1 = 0.34
D_max = 4
alpha = np.linspace(0, 1, 20+1)

comm = MPI.COMM_WORLD
cell_type = mesh.CellType.tetrahedron

domain = mesh.create_box(
    comm,
    [[0.0, 0.0, 0.0], [Lx, Ly, Lz]],
    [nx, ny, nz],
    cell_type=cell_type
)

In [3]:
C10 = fem.Constant(domain, default_scalar_type(C10))
C01 = fem.Constant(domain, default_scalar_type(C01))
D1 = fem.Constant(domain, default_scalar_type(D1))

B = fem.Constant(domain, default_scalar_type((0.0, 0.0, 0.0)))
T = fem.Constant(domain, default_scalar_type((0.0, 0.0, 0.0)))


In [4]:
V_space = fem.functionspace(domain, ("Lagrange", 1, (tdim,)))

# BC

def left_boundary(x): # Gamma_D0
    return np.isclose(x[0], 0.0)

def right_boundary(x): # Gamma_D1
    return np.isclose(x[0], Lx)

def every_other_boundary(x): # Gamma_N
    return np.isclose(x[1], 0.0) | np.isclose(x[1], Ly) | np.isclose(x[2], 0.0) | np.isclose(x[2], Lz)

left_dofs = fem.locate_dofs_geometrical(V_space, left_boundary)
right_dofs = fem.locate_dofs_geometrical(V_space, right_boundary)
bc1 = fem.dirichletbc(np.array([0.0, 0.0, 0.0], dtype=default_scalar_type), left_dofs, V_space)
#lo definire en cada paso mejor
bc2 = fem.dirichletbc(np.array([alpha[-1]*D_max, 0.0, 0.0], dtype=default_scalar_type), right_dofs, V_space)

other_facets = mesh.locate_entities_boundary(domain, tdim-1, every_other_boundary)
facet_tags = mesh.meshtags(domain, tdim-1, other_facets, np.full(other_facets.shape, 1, dtype=np.int32))
ds = ufl.Measure("ds", domain, subdomain_data=facet_tags)

In [5]:
def P_def(u, params: tuple, tdim=3):
    C10, C01, D1 = params
    I = ufl.Identity(tdim)
    F = ufl.variable(I + ufl.grad(u))
    J = ufl.det(F)
    C = ufl.dot(F.T, F)
    I1 = ufl.tr(C)
    I2 = 0.5 * (I1**2 - ufl.tr(ufl.dot(C, C)))
    
    I1bar = J**(-2/3) * I1
    I2bar = J**(-4/3) * I2
    
    W = C10 * (I1bar - 3) + C01 * (I2bar - 3) + 1/D1 * (J - 1)**2
    P = ufl.diff(W, F)
    return P

def residual(u, v, P, B, T):
    return ufl.inner(P(u), ufl.grad(v))*ufl.dx - ufl.dot(B,v)*ufl.dx - ufl.dot(T,v)*ds(1)

def P(u): return P_def(u, (C10, C01, D1), tdim=tdim)


In [6]:
# desde aqui puedo recorrer la simulacion
u = fem.Function(V_space)
du = ufl.TrialFunction(V_space)
v = ufl.TestFunction(V_space)

R = residual(u, v, P, B, T)
DR = ufl.derivative(R, u, du)

bcs = [bc1, bc2]
problem = NonlinearProblem(R, u, bcs=bcs, J=DR)

solver = NewtonSolver(comm, problem)
solver.rtol = 1e-7
solver.atol = 1e-9
solver.max_it = 30
solver.report = True
ksp = solver.krylov_solver
ksp.setType("preonly")
pc = ksp.getPC()
pc.setType("lu")
pc.setFactorSolverType("mumps")


In [7]:
I = ufl.Identity(tdim)
F = I + ufl.grad(u)
J = ufl.det(F)
_P = P(u)
sigma = ufl.dot(_P, F.T) / J

V_J = fem.functionspace(domain, ("DG", 0))
V_sigma = fem.functionspace(domain, ("DG", 0, (tdim, tdim)))

expr_J = fem.Expression(J, V_J.element.interpolation_points())
expr_sigma = fem.Expression(sigma, V_sigma.element.interpolation_points())


In [8]:
uhs = []
js = []
sigmas = []

for a in alpha:
    print(f"alpha = {a:.2f}", end=", ")
    bc2 = fem.dirichletbc(np.array([a*D_max, 0.0, 0.0], dtype=default_scalar_type), right_dofs, V_space)
    problem.bcs = [bc1, bc2]

    nit, flag = solver.solve(u)
    u.x.scatter_forward()
    print(f"Number of Newton iterations: {nit}, Converged: {flag}")
    
    u_copy = fem.Function(V_space)    
    J_func = fem.Function(V_J, name="J")
    sigma_func = fem.Function(V_sigma, name="Sigma")
    
    u_copy.x.array[:] = u.x.array[:]
    J_func.interpolate(expr_J)
    sigma_func.interpolate(expr_sigma)
    J_func.x.scatter_forward()
    sigma_func.x.scatter_forward()
    
    uhs.append(u_copy)
    js.append(J_func)
    sigmas.append(sigma_func)


alpha = 0.00, Number of Newton iterations: 0, Converged: True
alpha = 0.05, Number of Newton iterations: 3, Converged: True
alpha = 0.10, Number of Newton iterations: 3, Converged: True
alpha = 0.15, Number of Newton iterations: 3, Converged: True
alpha = 0.20, Number of Newton iterations: 3, Converged: True
alpha = 0.25, Number of Newton iterations: 3, Converged: True
alpha = 0.30, Number of Newton iterations: 3, Converged: True
alpha = 0.35, Number of Newton iterations: 3, Converged: True
alpha = 0.40, Number of Newton iterations: 2, Converged: True
alpha = 0.45, Number of Newton iterations: 2, Converged: True
alpha = 0.50, Number of Newton iterations: 2, Converged: True
alpha = 0.55, Number of Newton iterations: 2, Converged: True
alpha = 0.60, Number of Newton iterations: 2, Converged: True
alpha = 0.65, Number of Newton iterations: 2, Converged: True
alpha = 0.70, Number of Newton iterations: 2, Converged: True
alpha = 0.75, Number of Newton iterations: 2, Converged: True
alpha = 

---

In [9]:
pv.set_plot_theme("document")
pv.global_theme.colorbar_horizontal.width = 0.5
pv.global_theme.colorbar_horizontal.position_x = 0.25

topology, cell_types, geometry = plot.vtk_mesh(V_space)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

uhs = [uh.x.array.reshape((geometry.shape[0], 3)) for uh in uhs]
js = [j.x.array for j in js]
sigmas = [sigma.x.array.reshape(cell_types.shape[0], 3, 3) for sigma in sigmas]


In [10]:
topology, cell_types, geometry = plot.vtk_mesh(V_space)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

i = -1
grid["displacement"] = uhs[i]
grid["sigma"] = sigmas[i]
grid["J"] = js[i]

warped = grid.warp_by_vector("displacement")

pl = pv.Plotter(window_size=[1000, 500])
actor_0 = pl.add_mesh(grid, style="wireframe", color="black")
actor_1 = pl.add_mesh(warped, show_edges=True, edge_opacity=0.2, color="lightblue")#, cmap="viridis")

pl.view_isometric()
pl.camera.azimuth += -100
pl.camera.elevation -= 20
pl.camera.zoom(2.0)
pl.screenshot("assets/deformed_mesh.png", scale=2)
pl.show()


Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e69c424d0_0&reconnect=auto" class="pyvi…

In [11]:
u_magnitude = np.linalg.norm(warped["displacement"], axis=1)
warped.point_data["U_magnitude"] = u_magnitude

pl = pv.Plotter()

pl.add_mesh(warped, scalars="U_magnitude", show_edges=True, edge_opacity=0.2, cmap="viridis", scalar_bar_args={"title": "U_magnitude"})

pl.camera.parallel_projection = True
pl.view_xz()
pl.camera.zoom(1.5)
pl.add_axes()
pl.screenshot("assets/foto_2.png", scale=2)
pl.show()

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e8bf3bcd0_1&reconnect=auto" class="pyvi…

In [12]:
# Cortes
slice_xz = warped.slice(normal=(0, 1, 0), origin=(0, Ly/2, 0))
slice_xy = warped.slice(normal=(0, 0, 1), origin=(0, 0, Lz/2))
slices = [slice_xz, slice_xy]

pl = pv.Plotter()

pl.add_mesh(warped, color="lightgray", opacity=0.2, show_edges=True)

clim = [np.nanmin(warped["J"]), np.nanmax(warped["J"])]
for sli in slices:
    pl.add_mesh(sli, scalars="J", cmap="viridis", clim=clim, scalar_bar_args={"title": "Jacobian"})

pl.parallel_projection = True
pl.view_isometric()
pl.camera.azimuth += -100
pl.camera.elevation -= 20
pl.camera.zoom(1.50)
#pl.add_axes()
pl.screenshot("assets/J.png", scale=2)
pl.show()

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a740510_2&reconnect=auto" class="pyvi…

In [13]:
s_names = ["sxx", "sxy", "sxz", "syx", "syy", "syz", "szx", "szy", "szz"]
for i, s_name in enumerate(s_names):
    warped.cell_data[s_name] = warped["sigma"][:, i]

slice_xz = warped.slice(normal=(0, 1, 0), origin=(0, Ly/2, 0))
slice_xy = warped.slice(normal=(0, 0, 1), origin=(0, 0, Lz/2))


In [14]:
for s_name in s_names:
    pl = pv.Plotter(off_screen=True)
    slices = [slice_xz, slice_xy]
    pl.add_mesh(warped, color="lightgray", opacity=0.2, show_edges=True)

    clim = max(-np.nanmin(warped[s_name]), np.nanmax(warped[s_name])) * np.array([-1, 1])
    for sli in slices:
        pl.add_mesh(sli, scalars=s_name, cmap="coolwarm", clim=clim, scalar_bar_args={"title": s_name})

    pl.parallel_projection = True
    pl.view_isometric()
    pl.camera.azimuth += -100
    pl.camera.elevation -= 20
    pl.camera.zoom(1.50)
    #pl.add_axes()
    pl.screenshot(f"assets/{s_name}.png", scale=2)
    pl.show()


Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a7c4710_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a7424d0_4&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a7d4b50_5&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a5ccf90_6&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a5e4f90_7&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e3a48bd10_8&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e30b3c5d0_9&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e30bdf050_10&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:37873/index.html?ui=P_0x7f8e30a43110_11&reconnect=auto" class="pyv…

In [21]:
topology, cell_types, geometry = plot.vtk_mesh(V_space)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)


X = grid.points.copy()

bounds_min = X.min(axis=0)
bounds_max = X.max(axis=0)
for ui in uhs:
    X_def = X + ui
    bounds_min = np.minimum(bounds_min, X_def.min(axis=0))
    bounds_max = np.maximum(bounds_max, X_def.max(axis=0))

center = (bounds_min + bounds_max) / 2
lengths = bounds_max - bounds_min
max_length = lengths.max()

margin = 1.10
half_size = max_length * margin / 2


p = pv.Plotter(off_screen=True, window_size=[1000, 500])
#p.open_movie("assets/animation.avi", framerate=5)
p.open_gif("assets/animation.gif", fps=5)

p.add_mesh(grid, style="wireframe", color="black", opacity=0.3)

deformed_grid = grid.copy()
deformed_grid.points = X + uhs[0]

actor = p.add_mesh(deformed_grid, show_edges=True, edge_opacity=0.2, color="lightblue")

p.view_isometric()
p.camera.focal_point = center
p.camera.position = center + np.array([1.5, -2.0, 1.5]) * max_length
p.camera.up = (0, 0, 1)

p.camera.azimuth += -10
p.camera.elevation -= 10
p.camera.zoom(2.5)

for i, ui in enumerate(uhs):
    deformed_grid.points = X + ui

    deformed_grid.GetPoints().Modified()
    deformed_grid.Modified()
    p.add_text(f"alpha = {alpha[i]:.2f}", position="upper_right", font_size=14, name="dynamic_text")

    p.write_frame()
p.close()
